In [1]:
import requests
from tqdm import tqdm
import re
import pandas as pd 
import json

/Users/mqorbani3/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [86]:
# Correct endpoint information
endpoint = "https://eastus2.api.cognitive.microsoft.com"
deployment_id = "ls35-DE00012916-o1"
api_version = "2025-01-01-preview"
api_key = "0906319d295848d397f6420b9c2c59b1"

# endpoint = "https://eastus2.api.cognitive.microsoft.com"
# deployment_id = "ls35-DE00012916-gpt-5"
# api_version = "2025-04-01-preview" 
# api_key = "0906319d295848d397f6420b9c2c59b1"


# Test prompt
test_prompt = "Write a short, friendly email about prompt engineering."

# Headers and payload
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

data = {
    "messages": [
        {"role": "user", "content": test_prompt}
    ],
    "max_completion_tokens": 2500,
    #for gpt 4o and 4.1:  "max_tokens": 100,
    #"temperature": 0.7,
}

# Send the request
response = requests.post(
    f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
    headers=headers,
    json=data
)

# Show the result
try:
    result = response.json()
    if "choices" in result:
        print("IT WORKED! Here's the response:")
        print(result["choices"][0]["message"]["content"])
    else:
        print("Oops, something went wrong:")
        print(result)
except Exception as e:
    print("Failed to parse JSON response:", e)
    print("Raw response:", response.text)


IT WORKED! Here's the response:
Subject: Exploring the World of Prompt Engineering

Hello [Name],

I hope you’re doing well! I’ve recently been diving into the fascinating art of prompt engineering—crafting carefully worded instructions that guide AI models toward more accurate, useful responses. It’s amazing how even small tweaks can reshape the outcomes.

If you’re curious to learn more about creating effective prompts and best practices for working with AI, I’d love to chat or share resources. Let me know if you’d be interested!

Best regards,
[Your Name]


# Model 4 (has task duration, and discription, current task step, hand actions data)

In [87]:
import ast
def build_prompt_full_history(task_type, task_duration, current_question, current_time, task_step, qa_history=[], task_description="", hand_actions=None, hand_actions_now=None):
    """
    hand_actions: list of [verb, noun, task_step]
    qa_history: list of tuples (question_time, question, answer)
    current_time: float (in seconds)
    task_description: str, describing the typical steps involved in the task
    """

    # Filter Q&A before current time
    previous_qa = [(t, q, a) for t, q, a in qa_history if t < current_time]

    prompt = f"""You are an instructor standing beside a student who is performing the task: {task_type}.

This task usually follows these common steps (though the exact sequence may vary slightly across different sessions):
{task_description}

The total duration of the task is approximately {int(task_duration)} seconds.
"""

    if previous_qa:
        prompt += f"""\nHere is the full conversation history up to {int(current_time)} seconds:\n"""
        for t, q, a in previous_qa:
            prompt += f"""Student: "{q}" at time {t} seconds \nInstructor: "{a}"\n\n"""
    else:
        prompt += "\n\nThere has been no prior student question so far. This is the first question.\n"

    if task_step == "No coarse-grained action":
        prompt += "The current step of the task is not specified. It may be that the student has not yet started the task, or the step detection model may have failed to identify the current step.\n"
    else:
        prompt += f"The current step of the task is: {task_step}\n"

    # Add hand actions (grouped by step)
    
    from collections import defaultdict
    step_to_actions = defaultdict(list)


    raw_data = hand_actions
    if isinstance(raw_data, str):
        hand_actions = ast.literal_eval(raw_data)
    else:
        hand_actions = raw_data  # already parsed

    
    if len(hand_actions) == 0:
        prompt += f"\nRecent hand actions up to last question: No hand actions detected so far.\n"

    else:
        for action in list(hand_actions):
           
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nRecent hand actions up to the last student question:\n"
        prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            # if step == "No coarse-grained action":
            #     step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"{actions_str}"

    step_to_actions = defaultdict(list)
    raw_data = hand_actions_now
    if isinstance(raw_data, str):
        hand_actions_now = ast.literal_eval(raw_data)
    else:
        hand_actions_now = raw_data  # already parsed

    
    if len(hand_actions_now) == 0:
        prompt += f"\nMost Recent hand actions after last question (if any) was asked up to {int(current_time)} seconds (time of the current question): No hand actions detected after last question so far.\n"

    else:
        for action in list(hand_actions_now):
           
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nMost Recent hand actions after last question was asked up to {int(current_time)} seconds (time of the current question):\n"
        # prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            # if step == "No coarse-grained action":
            #     step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"{actions_str}"


        

    prompt += f"""\n
Now, the student asks (at {int(current_time)} seconds):  
Student: "{current_question}"  

Please respond like a real-time instructor: clear, concise, short, and based on the context you have. Pay close attention to the hand actions data, as it offers real-time insights leading up to the moment the question was asked.
"""
    return prompt


In [88]:
df = pd.read_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/combined_question_action_summary.csv')
df['response_from_ai'] = ""
no_answers_rows = []

In [89]:
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# Initialize column if not present
if 'response_from_ai' not in df.columns:
    df['response_from_ai'] = ""

for idx in tqdm(df.index):
    if idx % 50 == 0:
        print(f"Processing row {idx}...")

    row = df.loc[idx]
    video = row['video_name']
    current_time = row['question_time']

    if pd.isna(current_time):
        continue  # skip if no time info

    # Get all previous Q&A for this video before this time
    earlier_rows = df[
        (df['video_name'] == video) &
        (df['question_time'] < current_time)
    ]

    # Use already filled responses
    qa_history = list(zip(
        earlier_rows['question_time'],
        earlier_rows['question'],
        earlier_rows['response_from_ai']
    ))

    # Collect all previous hand actions into a single list
    all_prev_hand_actions = []
    for _, r in earlier_rows.iterrows():
        if not pd.isna(r['hand_actions_data']):
            try:
                ha = ast.literal_eval(r['hand_actions_data']) if isinstance(r['hand_actions_data'], str) else r['hand_actions_data']
                all_prev_hand_actions.extend(ha)
            except Exception as e:
                print(f"❌ Error parsing hand actions at row {r.name}: {e}")


    # Step 2: Parse current row's hand actions
    hand_actions_now = []
    if not pd.isna(row['hand_actions_data']):
        try:
            hand_actions_now = ast.literal_eval(row['hand_actions_data']) if isinstance(row['hand_actions_data'], str) else row['hand_actions_data']
        except Exception as e:
            print(f"❌ Error parsing current hand actions at row {idx}: {e}")
            hand_actions_now = []

    # Step 3: Combine them
    # combined_hand_actions = all_prev_hand_actions + hand_actions_now

    # Build prompt
    prompt = build_prompt_full_history(
        task_type=row['task_type'],
        task_duration=row['task_duration'],
        current_question=row['question'],
        current_time=current_time,
        task_step = row['task_step'],
        qa_history=qa_history,
        task_description = row['task_description'],
        hand_actions= all_prev_hand_actions,
        hand_actions_now = hand_actions_now
    )
    # print(all_prev_hand_actions)
    # print(hand_actions_now)
    # print(prompt)
    # print("++++++++++++++++++++++++++++++++")

    # Send to GPT API
    data = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_completion_tokens": 2500,
        # "temperature": 0.2,
        # "max_tokens": 500
    }

    try:
        response = requests.post(
            f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
            headers=headers,
            json=data
        )

        result = response.json()
        if ("choices" in result and 
            "message" in result["choices"][0] and 
            "content" in result["choices"][0]["message"]):

            ai_reply = result["choices"][0]["message"]["content"]
            ai_reply = re.sub(r'^\s*Instructor\s*:\s*', '', ai_reply, flags=re.IGNORECASE)
            df.at[idx, "response_from_ai"] = ai_reply

        else:
            print(f"⚠️ Unexpected API response at row {idx}:\n{result}")
            df.at[idx, "response_from_ai"] = "[Error: Unexpected structure]"

            no_answers_rows.append(idx)

            # Save malformed result to a JSON file
            # with open(f"failed_response_row_{idx}.json", "w") as f:
            #     json.dump(result, f, indent=2)


    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        df.at[idx, "response_from_ai"] = "[Error: API call failed]"


  0%|          | 0/640 [00:00<?, ?it/s]

Processing row 0...


  2%|▏         | 11/640 [03:16<2:33:41, 14.66s/it]

⚠️ Unexpected API response at row 10:
{'choices': [{'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': True, 'severity': 'medium'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}, 'finish_reason': 'content_filter', 'index': 0, 'logprobs': None, 'message': {'annotations': [], 'refusal': None, 'role': 'assistant'}}], 'created': 1756750082, 'id': 'chatcmpl-CB3EgvJTjULdJtnmW5toHvXx5NAlJ', 'model': 'o1-2024-12-17', 'object': 'chat.completion', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'system_fingerprint': 'fp_a2dc940e6b', 'usage': {'completion_tokens': 378, 'completion_tokens_details': {'accepte

  2%|▏         | 13/640 [03:28<1:43:08,  9.87s/it]

...

Processing row 50...


 16%|█▌        | 100/640 [54:01<2:04:54, 13.88s/it] 

Processing row 100...


 16%|█▌        | 102/640 [1:57:27<10:19:30, 69.09s/it]


KeyboardInterrupt: 

In [183]:
for idx in tqdm(df.index):
    if idx == 541:
        if idx % 50 == 0:
            print(f"Processing row {idx}...")

        row = df.loc[idx]
        video = row['video_name']
        current_time = row['question_time']

        if pd.isna(current_time):
            continue  # skip if no time info

        # Get all previous Q&A for this video before this time
        earlier_rows = df[
            (df['video_name'] == video) &
            (df['question_time'] < current_time)
        ]

        # Use already filled responses
        qa_history = list(zip(
            earlier_rows['question_time'],
            earlier_rows['question'],
            earlier_rows['response_from_ai']
        ))

        # Collect all previous hand actions into a single list
        all_prev_hand_actions = []
        for _, r in earlier_rows.iterrows():
            if not pd.isna(r['hand_actions_data']):
                try:
                    ha = ast.literal_eval(r['hand_actions_data']) if isinstance(r['hand_actions_data'], str) else r['hand_actions_data']
                    all_prev_hand_actions.extend(ha)
                except Exception as e:
                    print(f"❌ Error parsing hand actions at row {r.name}: {e}")


        # Step 2: Parse current row's hand actions
        hand_actions_now = []
        if not pd.isna(row['hand_actions_data']):
            try:
                hand_actions_now = ast.literal_eval(row['hand_actions_data']) if isinstance(row['hand_actions_data'], str) else row['hand_actions_data']
            except Exception as e:
                print(f"❌ Error parsing current hand actions at row {idx}: {e}")
                hand_actions_now = []

        # Step 3: Combine them
        # combined_hand_actions = all_prev_hand_actions + hand_actions_now

        # Build prompt
        prompt = build_prompt_full_history(
            task_type=row['task_type'],
            task_duration=row['task_duration'],
            current_question=row['question'],
            current_time=current_time,
            task_step = row['task_step'],
            qa_history=qa_history,
            task_description = row['task_description'],
            hand_actions= all_prev_hand_actions,
            hand_actions_now = hand_actions_now
        )
        # print(all_prev_hand_actions)
        # print(hand_actions_now)
        # print(prompt)
        # print("++++++++++++++++++++++++++++++++")

        # Send to GPT API
        data = {
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_completion_tokens": 1500,
            # "temperature": 0.2,
            # "max_tokens": 500
        }

        try:
            response = requests.post(
                f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
                headers=headers,
                json=data
            )

            result = response.json()
            if ("choices" in result and 
                "message" in result["choices"][0] and 
                "content" in result["choices"][0]["message"]):

                ai_reply = result["choices"][0]["message"]["content"]
                ai_reply = re.sub(r'^\s*Instructor\s*:\s*', '', ai_reply, flags=re.IGNORECASE)
                df.at[idx, "response_from_ai"] = ai_reply

            else:
                print(f"⚠️ Unexpected API response at row {idx}:\n{result}")
                df.at[idx, "response_from_ai"] = "[Error: Unexpected structure]"

                no_answers_rows.append(idx)

                # Save malformed result to a JSON file
                # with open(f"failed_response_row_{idx}.json", "w") as f:
                #     json.dump(result, f, indent=2)


        except Exception as e:
            print(f"❌ Error at row {idx}: {e}")
            df.at[idx, "response_from_ai"] = "[Error: API call failed]"


100%|██████████| 640/640 [00:13<00:00, 46.77it/s]


In [184]:
df.iloc[541,:]

video_name                                          z059-june-28-22-switch
task_type                                                     setup switch
task_duration                                                          255
task_description         1. Detach the Joy-Con controllers from the Nin...
task_step                                         No coarse-grained action
hand_actions_data                                                       []
hints_before_question                                                   []
question                                                        This ones?
question_time                                                      213.677
instructor_answer                                                     yes.
response_from_ai                                                          
Name: 541, dtype: object

In [180]:
df.to_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/o1/model4_new_results.csv')

# Evaluation

In [330]:
dff = df.copy()
dff.head(5)

,video_name,task_type,task_duration,task_description,task_step,hand_actions_data,hints_before_question,question,question_time,instructor_answer,response_from_ai
0,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""dslr"", ""The student detaches the le...","[[1.264, ""We will now operate the DSLR, and th...",Could you repeat please?,58.329,The next step is to detach the lens cover.,"Sure. So far, you’ve picked up the camera, pre..."
1,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""sd_slot"", ""The student detaches the...","[[69.1, ""Nice.""], [69.975, ""Now please attach ...",I'm sorry?,159.200,You can now stop the recording.,No worries. We just finished swapping the SD c...
2,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The student detaches the lens of the camera.,"[[""grab"", ""dslr"", ""The student detaches the le...","[[0.887, ""Now we are going to look at the DSLR...",To detach what?,16.300,To detach the lens you should press on the but...,You’re detaching the camera lens from the DSLR...
3,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The students changes the battery of the dslr.,"[[""turn"", ""dslr"", ""The student detaches the le...","[[27.2, ""The lens is um, the long round thing,...",Change the battery?,140.100,Yeah.,Yes—go ahead and insert the fresh battery. Mak...
4,R159-3Oct-RAM,assemble computer,223,The task involves assembling or replacing comp...,No coarse-grained action,[],"[[6.153, ""So the first step is to replace the ...",To replace the RAM card?,10.200,Yep.,"Yes, let’s go ahead and replace the RAM. First..."


In [332]:
dff = pd.read_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/o1/model4_results.csv')

In [333]:
from bert_score import score
from sentence_transformers import SentenceTransformer, util
import pandas as pd
from tqdm import tqdm


# Load models
bert_model = "microsoft/deberta-xlarge-mnli"  # Better BERTScore performance
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize new columns
dff['bertscore_precision'] = None
dff['bertscore_recall'] = None
dff['bertscore_f1'] = None
dff['semantic_similarity'] = None

# Compute metrics row by row
for i in tqdm(dff.index):
    candidate = str(dff.at[i, 'response_from_ai'])
    reference = str(dff.at[i, 'instructor_answer'])

    # Skip empty values
    if not candidate.strip() or not reference.strip():
        continue

    # --- BERTScore ---
    P, R, F1 = score([candidate], [reference], lang='en', model_type=bert_model, verbose=False)
    dff.at[i, 'bertscore_precision'] = round(P.item(), 4)
    dff.at[i, 'bertscore_recall'] = round(R.item(), 4)
    dff.at[i, 'bertscore_f1'] = round(F1.item(), 4)

    # --- SentenceTransformer Similarity ---
    emb1 = st_model.encode(candidate, convert_to_tensor=True)
    emb2 = st_model.encode(reference, convert_to_tensor=True)
    sim_score = util.pytorch_cos_sim(emb1, emb2).item()
    dff.at[i, 'semantic_similarity'] = round(sim_score, 4)


100%|██████████| 640/640 [34:43<00:00,  3.26s/it]


In [335]:
dff

,Unnamed: 0,video_name,task_type,task_duration,task_description,task_step,hand_actions_data,hints_before_question,question,question_time,instructor_answer,response_from_ai,bertscore_precision,bertscore_recall,bertscore_f1,semantic_similarity
0,0,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""dslr"", ""The student detaches the le...","[[1.264, ""We will now operate the DSLR, and th...",Could you repeat please?,58.329000,The next step is to detach the lens cover.,"Sure. So far, you’ve picked up the camera, pre...",0.578,0.6897,0.6289,0.6345
1,1,z198-sep-18-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,No coarse-grained action,"[[""grab"", ""sd_slot"", ""The student detaches the...","[[69.1, ""Nice.""], [69.975, ""Now please attach ...",I'm sorry?,159.200000,You can now stop the recording.,No worries. We just finished swapping the SD c...,0.5111,0.6294,0.5641,0.2954
2,2,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The student detaches the lens of the camera.,"[[""grab"", ""dslr"", ""The student detaches the le...","[[0.887, ""Now we are going to look at the DSLR...",To detach what?,16.300000,To detach the lens you should press on the but...,You’re detaching the camera lens from the DSLR...,0.721,0.719,0.72,0.7442
3,3,z073-july-01-22-dslr,setup camera,166,1. Inspect the DSLR camera. 2. Detach the lens...,The students changes the battery of the dslr.,"[[""turn"", ""dslr"", ""The student detaches the le...","[[27.2, ""The lens is um, the long round thing,...",Change the battery?,140.100000,Yeah.,Yes—go ahead and insert the fresh battery. Mak...,0.3053,0.7988,0.4418,0.077
4,4,R159-3Oct-RAM,assemble computer,223,The task involves assembling or replacing comp...,No coarse-grained action,[],"[[6.153, ""So the first step is to replace the ...",To replace the RAM card?,10.200000,Yep.,"Yes, let’s go ahead and replace the RAM. First...",0.2734,0.7851,0.4055,0.0377
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,635,z055-june-27-22-knarrevik_assemble,assemble nightstand,732,1. Inspect the components of the nightstand (e...,The student mounts the low piece of furniture.,"[[""grab"", ""tray"", ""The student mounts the low ...","[[10.131, ""Ah. First, you need to take a while...",This one.,38.619608,Yes.,"Yes, that’s the correct leg. Line it up with t...",0.3866,0.8059,0.5225,0.0922
636,636,z055-june-27-22-knarrevik_assemble,assemble nightstand,732,1. Inspect the components of the nightstand (e...,The student tightens the first side of the fur...,"[[""lift"", ""tray"", ""The student mounts the low ...","[[46.096, ""Yes.""], [47.792, ""Yeah.""], [57.775,...",Tighten everything now.,504.275000,Yes.,"Yes, go ahead and tighten all the screws in a ...",0.3408,0.8281,0.4828,0.0554
637,637,z142-aug-17-22-printer_big,setup big printer,109,1. Turn on the printer (using the appropriate ...,The student exchanges an black ink cartridge.,"[[""press"", ""button"", ""The student turns on the...","[[2.1, ""Ok, please look at the big printer.""],...",This one?,63.400000,"The first top one, white door on the very top.","Yes, that’s the black cartridge. Go ahead and ...",0.6094,0.6162,0.6128,0.1449
638,638,z142-aug-17-22-printer_big,setup big printer,109,1. Turn on the printer (using the appropriate ...,The student exchanges an black ink cartridge.,"[[""approach"", ""printer"", ""The student exchange...","[[67.1, ""The student can not find the cartridg...",This?,68.100000,"No, the white color on the top of the drawer.","Yes, that's the black cartridge. Gently pull i...",0.5911,0.6795,0.6322,0.3195


In [336]:
dff.to_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/o1/model4_results_with_metrics.csv', index=False)

# LLM jUDGE

The first ten rows are used to evaluate the judgement of llm, so we will not consider them in the final evaluation of our model!

In [4]:
import urllib.parse

safe_url = "https://nam12.safelinks.protection.outlook.com/?url=https%3A%2F%2Feastus2.api.cognitive.microsoft.com%2Fopenai%2Fdeployments%2Fls35-DE00012916-o3%2Fchat%2Fcompletions%3Fapi-version%3D2025-01-01-preview&data=05%7C02%7Cmqorbani3%40gatech.edu%7Cac450bd787ed4067d1c908dd97179a27%7C482198bbae7b4b258b7a6d7f32faa083%7C1%7C0%7C638832849805847242%7CUnknown%7CTWFpbGZsb3d8eyJFbXB0eU1hcGkiOnRydWUsIlYiOiIwLjAuMDAwMCIsIlAiOiJXaW4zMiIsIkFOIjoiTWFpbCIsIldUIjoyfQ%3D%3D%7C0%7C%7C%7C&sdata=BnqxtdKbnW03%2F5TU4UyXEwUXksjB7xjlTfiI2lKsLHQ%3D&reserved=0"
parsed = urllib.parse.urlparse(safe_url)
actual_url = urllib.parse.parse_qs(parsed.query)["url"][0]
print("Decoded URL:", urllib.parse.unquote(actual_url))

Decoded URL: https://eastus2.api.cognitive.microsoft.com/openai/deployments/ls35-DE00012916-o3/chat/completions?api-version=2025-01-01-preview


In [337]:
# import pandas as pd
# import requests
# import re
# import json
# from tqdm import tqdm

# Load your dataset
dfff = pd.read_csv("/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/o1/model4_results_with_metrics.csv")

# API config
api_key = "0906319d295848d397f6420b9c2c59b1"
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}
endpoint = "https://eastus2.api.cognitive.microsoft.com"
deployment_id = "ls35-DE00012916-gpt-4o"
api_version = "2025-01-01-preview"

# Initialize columns for judgment if not present
judgment_cols = [
    "judge_correctness", "judge_completeness", "judge_relevance", "judge_clarity",
    "judge_final_score", "judge_justification"
]
for col in judgment_cols:
    if col not in dfff.columns:
        dfff[col] = None

# LLM Judge Prompt Template
def build_judge_prompt(task_type, task_duration, task_description, question_time, conversation_history, task_step, hand_actions, question, instructor_answer, ai_answer):
    prompt = f"""You are an expert judge evaluating the quality of an AI instructor’s response to a student’s question during a technical task. Your evaluation must consider:

    - The ground truth instructor response
    - The AI-generated response
    - The task context, including task type, common task steps (though the exact sequence may vary slightly across different sessions), total duration, conversation history, 
    the current task step, and hand actions data (Pay close attention to the hand actions data, as it offers real-time insights leading up to the moment the question was asked.)
    - The student’s question and the time it was asked

    Your goal is to determine whether the AI’s response is correct, complete, contextually appropriate, and clearly communicated. The ground truth instructor response is not always perfect, so you should not expect the AI to match it exactly. Instead, focus on whether the AI’s response is a reasonable and helpful answer to the student’s question given the context provided.

    ---

    Task Type: {task_type}  
    Total Task Duration: {int(task_duration)} seconds  
    Task Steps: {task_description}   
    Time of Student Question: {int(question_time)} seconds  

    Conversation History (Previous Student Questions & Instructor Answers):  
    {conversation_history}\n\n"""


    if task_step == "No coarse-grained action":
            prompt += f"The current step of the task is not specified. It may be that the student has not yet started the task, or the step detection model may have failed to identify the current step.\n"

    else:
        prompt += f"The current step of the task is ({task_step})\n"

    # current task step: {task_description.split('.')[0]} (if applicable)

        # Add hand actions (grouped by step)
    from collections import defaultdict
    step_to_actions = defaultdict(list)


    raw_data = hand_actions
    if isinstance(raw_data, str):
        hand_actions = ast.literal_eval(raw_data)
    else:
        hand_actions = raw_data  # already parsed

     
    
    if len(hand_actions) == 0:
        prompt += "\nRecent hand actions grouped by task step: No hand actions detected so far.\n"

    else:
        for action in list(hand_actions):
           
            
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nRecent hand actions grouped by task step up to {int(current_time)} seconds (time of the current question):\n"
        prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            if step == "No coarse-grained action":
                step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"Step {idx} ({step}): {actions_str}\n"


    prompt += f"""
    ---

    Current Student Question:  
    {question}

    Ground Truth Instructor Response:  
    {instructor_answer}

    AI-Generated Response:  
    {ai_answer}

    ---

    ### Evaluation Criteria (Rate each on a scale from 0–5):

    - Correctness: 5 = Fully accurate, 3 = Minor issues, 1 = Incorrect, 0 = Wrong
    - Completeness: 5 = Fully answers, 3 = Partially, 1 = Minimal, 0 = None
    - Contextual Relevance: 5 = Fully grounded, 3 = Weakly grounded, 1 = Irrelevant, 0 = Off-topic
    - Clarity: 5 = Very clear, 3 = Somewhat clear, 1 = Vague, 0 = Incomprehensible

    Then provide your overall assessment:

    - Final Score (0–5): Your holistic judgment of the AI response (not an average of the above scores).
    - Justification: 2–3 sentence explanation

    Return your evaluation in this exact format:
    Correctness: X/5  
    Completeness: X/5  
    Contextual Relevance: X/5  
    Clarity: X/5  
    Final Score: X/5  
    Justification: ..."""
    
    return prompt

# Generate conversation history
def build_conversation_history(qa_history, current_time):
    return "\n".join(
        f'Student: "{q}" at {t} sec\nInstructor: "{a}"'
        for t, q, a in qa_history if t < current_time
    ) or "No prior conversation. This is the first question."

# Evaluate AI responses using LLM judge
for idx in tqdm(dfff.index):
    # if idx <=10:
    try:
        row = dfff.loc[idx]
        if pd.isna(row["question_time"]) or pd.isna(row["response_from_ai"]):
            continue

        video = row["video_name"]
        current_time = row["question_time"]
        earlier_rows = dfff[(dfff["video_name"] == video) & (dfff["question_time"] < current_time)]

        qa_history = list(zip(
            earlier_rows["question_time"],
            earlier_rows["question"],
            earlier_rows["response_from_ai"]
        ))

        history = build_conversation_history(qa_history, current_time)

        judge_prompt = build_judge_prompt(
            task_type=row["task_type"],
            task_duration=row["task_duration"],
            task_description=row["task_description"],
            question_time=current_time,
            conversation_history=history,
            task_step=row["task_step"],
            hand_actions=row["hand_actions_data"],
            question=row["question"],
            instructor_answer=row["instructor_answer"],
            ai_answer=row["response_from_ai"]
        )
        # print(judge_prompt)
        # # API Call
        data = {
            "messages": [
                {"role": "user", "content": judge_prompt}
            ],
            "temperature": 0.2,
            "max_tokens": 500
        }

        response = requests.post(
            f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
            headers=headers,
            json=data
        )

        result = response.json()
        if ("choices" in result and
            "message" in result["choices"][0] and
            "content" in result["choices"][0]["message"]):

            content = result["choices"][0]["message"]["content"]

            match = re.search(r"""
                    Correctness:\s*(\d)\s*/\s*5\s*
                    .*?Completeness:\s*(\d)\s*/\s*5\s*
                    .*?Contextual\s+Relevance:\s*(\d)\s*/\s*5\s*
                    .*?Clarity:\s*(\d)\s*/\s*5\s*
                    .*?Final\s+Score:\s*(\d+(?:\.\d+)?)\s*/\s*5\s*
                    .*?Justification:\s*(.*)
                """, content, re.DOTALL | re.VERBOSE)

            if match:
                dfff.at[idx, "judge_correctness"] = float(match.group(1))
                dfff.at[idx, "judge_completeness"] = float(match.group(2))
                dfff.at[idx, "judge_relevance"] = float(match.group(3))
                dfff.at[idx, "judge_clarity"] = float(match.group(4))
                dfff.at[idx, "judge_final_score"] = float(match.group(5))
                dfff.at[idx, "judge_justification"] = match.group(6).strip()
                
            else:
                print(f"⚠️ Parsing failed at row {idx}")
                dfff.at[idx, "judge_justification"] = "[Parse error]"

    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        dfff.at[idx, "judge_justification"] = "[API error]"

# Save output
# df.to_csv("judged_responses.csv", index=False)
# print("✅ Judgments saved to judged_responses.csv")


100%|██████████| 640/640 [25:49<00:00,  2.42s/it]


In [ ]:
# Check the judgment results on first 10 rows
# dfff.iloc[0:10].to_csv("/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/test.csv", index=False)

In [338]:
dfff.to_csv('/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/o1/model4_results_with_metrics_and_LLMjudgments2.csv', index=False)

In [ ]:
#try another prompt

In [6]:
# import pandas as pd
# import requests
# import re
# import json
# from tqdm import tqdm
import ast
# Load your dataset
dfff = pd.read_csv("/Users/mqorbani3/Library/Mobile Documents/com~apple~CloudDocs/research/AI:VR/undergraduate researcher/preparing data/Prepared Data/Prompts/model4_results_with_metrics.csv")

# API config
api_key = "0906319d295848d397f6420b9c2c59b1"
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}
endpoint = "https://eastus2.api.cognitive.microsoft.com"
deployment_id = "ls35-DE00012916-gpt-4o"
api_version = "2025-01-01-preview"

# Initialize columns for judgment if not present
judgment_cols = [
    "judge_correctness", "judge_completeness", "judge_relevance", "judge_clarity",
    "judge_final_score", "judge_justification"
]
for col in judgment_cols:
    if col not in dfff.columns:
        dfff[col] = None

# LLM Judge Prompt Template
def build_judge_prompt(task_type, task_duration, task_description, question_time, conversation_history, task_step, hand_actions, question, instructor_answer, ai_answer):
    prompt = f"""You are an expert judge evaluating the quality of an AI instructor’s response to a student’s question during a technical task. Your evaluation must consider:

    - The ground truth instructor response
    - The AI-generated response
    - The task context, including task type, common task steps (though the exact sequence may vary slightly across different sessions), total duration, conversation history, 
    the current task step, and hand actions data (Pay close attention to the hand actions data, as it offers real-time insights leading up to the moment the question was asked.)
    - The student’s question and the time it was asked

    Your goal is to determine whether the AI’s response is correct, complete, contextually appropriate, and clearly communicated. The ground truth instructor response is not always perfect, so you should not expect the AI to match it exactly. Instead, focus on whether the AI’s response is a reasonable and helpful answer to the student’s question given the context provided.

    ---

    Task Type: {task_type}  
    Total Task Duration: {int(task_duration)} seconds  
    Task Steps: {task_description}   
    Time of Student Question: {int(question_time)} seconds  

    Conversation History (Previous Student Questions & Instructor Answers):  
    {conversation_history}\n\n"""


    if task_step == "No coarse-grained action":
            prompt += f"The current step of the task is not specified. It may be that the student has not yet started the task, or the step detection model may have failed to identify the current step.\n"

    else:
        prompt += f"The current step of the task is ({task_step})\n"

    # current task step: {task_description.split('.')[0]} (if applicable)

        # Add hand actions (grouped by step)
    from collections import defaultdict
    step_to_actions = defaultdict(list)


    raw_data = hand_actions
    if isinstance(raw_data, str):
        hand_actions = ast.literal_eval(raw_data)
    else:
        hand_actions = raw_data  # already parsed

     
    
    if len(hand_actions) == 0:
        prompt += "\nRecent hand actions grouped by task step: No hand actions detected so far.\n"

    else:
        for action in list(hand_actions):
           
            
            if isinstance(action, (list, tuple)) and len(action) == 3:
                verb, noun, step = action
                
                step_to_actions[step].append(f"{verb} {noun}, ")

        prompt += f"\nRecent hand actions grouped by task step up to {int(current_time)} seconds (time of the current question):\n"
        prompt += "(Each action is detected per frame in the form of a verb–noun pair and was extracted using a state-of-the-art hand action recognition model.)\n"
       
        for idx, (step, actions) in enumerate(step_to_actions.items(), 1):
            if step == "No coarse-grained action":
                step = "This step of the task is not specified"
            
            actions_str = ' '.join(actions)
            prompt += f"Step {idx} ({step}): {actions_str}\n"


    prompt += f"""
    ---

    Current Student Question:  
    {question}

    Ground Truth Instructor Response:  
    {instructor_answer}

    AI-Generated Response:  
    {ai_answer}

    ---

    ### Evaluation Criteria (Rate each on a scale from 0–5):

    #### 🔹 Correctness
    Is the response factually accurate and free from errors?
    - **5** = Fully accurate; no factual issues.
    - **4** = Mostly accurate; contains one very minor factual inaccuracy.
    - **3** = Minor factual issues, but the main idea is preserved.
    - **2** = Noticeable inaccuracies that reduce trust in the response.
    - **1** = Mostly incorrect or misinterprets the core idea.
    - **0** = Entirely wrong, harmful, or factually nonsensical.

    #### 🔹 Completeness
    Does it cover all essential aspects of the ground truth?
    - **5** = Fully comprehensive and covers all major points.
    - **4** = Covers most important content with slight omissions.
    - **3** = Partially complete; some key information missing.
    - **2** = Incomplete and misses several important components.
    - **1** = Covers only a small fraction of the necessary points.
    - **0** = No relevant information provided.

    #### 🔹 Contextual Relevance
    Is it tailored to the provided context and task?
    - **5** = Entirely relevant and specific to the given context.
    - **4** = Mostly relevant with a few general or unrelated parts.
    - **3** = Somewhat relevant but includes generic or off-topic content.
    - **2** = Only slightly related to the task or context.
    - **1** = Mostly off-topic or contextually incorrect.
    - **0** = Irrelevant or ignores the provided context entirely.

    #### 🔹 Clarity
    Is it well-structured, coherent, and easy to follow?
    - **5** = Exceptionally clear, fluent, and natural.
    - **4** = Clear and understandable with minor style issues.
    - **3** = Understandable but has awkward phrasing or structure.
    - **2** = Difficult to follow due to poor organization or grammar.
    - **1** = Disorganized or confusing.
    - **0** = Incoherent or unreadable.

    ---

    ### Final Judgment

    - **Final Score (0–5)**: Your holistic judgment of the AI response (not an average of the above scores). This score should align with your impression of how useful and high-quality the model’s response is as a real-time assistant.
    - Justification: 2–3 sentence explanation

    ---

    ### Example Scoring Scale (to reduce ambiguity):

    - **5**: As good or better than the human answer; highly helpful and natural.
    - **4**: Minor flaws or omissions, but still solid and useful.
    - **3**: Understandable but lacking in coverage or specificity.
    - **2**: Incorrect or confusing in several places.
    - **1**: Misleading, off-topic, or unusable.

    ---

    Return your evaluation in this exact format:
    Correctness: X/5  
    Completeness: X/5  
    Contextual Relevance: X/5  
    Clarity: X/5  
    Final Score: X/5  
    Justification: ..."""
    
    return prompt

# Generate conversation history
def build_conversation_history(qa_history, current_time):
    return "\n".join(
        f'Student: "{q}" at {t} sec\nInstructor: "{a}"'
        for t, q, a in qa_history if t < current_time
    ) or "No prior conversation. This is the first question."

# Evaluate AI responses using LLM judge
for idx in tqdm(dfff.index):
    # if idx <=10:
    try:
        row = dfff.loc[idx]
        if pd.isna(row["question_time"]) or pd.isna(row["response_from_ai"]):
            continue

        video = row["video_name"]
        current_time = row["question_time"]
        earlier_rows = dfff[(dfff["video_name"] == video) & (dfff["question_time"] < current_time)]

        qa_history = list(zip(
            earlier_rows["question_time"],
            earlier_rows["question"],
            earlier_rows["response_from_ai"]
        ))

        history = build_conversation_history(qa_history, current_time)

        judge_prompt = build_judge_prompt(
            task_type=row["task_type"],
            task_duration=row["task_duration"],
            task_description=row["task_description"],
            question_time=current_time,
            conversation_history=history,
            task_step=row["task_step"],
            hand_actions=row["hand_actions_data"],
            question=row["question"],
            instructor_answer=row["instructor_answer"],
            ai_answer=row["response_from_ai"]
        )
        # print(judge_prompt)
        # # API Call
        data = {
            "messages": [
                {"role": "user", "content": judge_prompt}
            ],
            "temperature": 0.2,
            "max_tokens": 500
        }

        response = requests.post(
            f"{endpoint}/openai/deployments/{deployment_id}/chat/completions?api-version={api_version}",
            headers=headers,
            json=data
        )

        result = response.json()
        if ("choices" in result and
            "message" in result["choices"][0] and
            "content" in result["choices"][0]["message"]):

            content = result["choices"][0]["message"]["content"]

            match = re.search(r"""
                    Correctness:\s*(\d)\s*/\s*5\s*
                    .*?Completeness:\s*(\d)\s*/\s*5\s*
                    .*?Contextual\s+Relevance:\s*(\d)\s*/\s*5\s*
                    .*?Clarity:\s*(\d)\s*/\s*5\s*
                    .*?Final\s+Score:\s*(\d+(?:\.\d+)?)\s*/\s*5\s*
                    .*?Justification:\s*(.*)
                """, content, re.DOTALL | re.VERBOSE)

            if match:
                dfff.at[idx, "judge_correctness"] = float(match.group(1))
                dfff.at[idx, "judge_completeness"] = float(match.group(2))
                dfff.at[idx, "judge_relevance"] = float(match.group(3))
                dfff.at[idx, "judge_clarity"] = float(match.group(4))
                dfff.at[idx, "judge_final_score"] = float(match.group(5))
                dfff.at[idx, "judge_justification"] = match.group(6).strip()
                
            else:
                print(f"⚠️ Parsing failed at row {idx}")
                dfff.at[idx, "judge_justification"] = "[Parse error]"

    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        dfff.at[idx, "judge_justification"] = "[API error]"

# Save output
# df.to_csv("judged_responses.csv", index=False)
# print("✅ Judgments saved to judged_responses.csv")


100%|██████████| 640/640 [18:23<00:00,  1.72s/it]


In [7]:
dfff.to_csv("model4_results_with_metrics_and_LLMjudgments3.csv", index=False)